# plot_03c. Forced response figures: the whole distribution, additivity and zonal means

The figures for the forced response, drawn from what notebook calc_02 saved, so nothing slow runs here. Notebook calc_02 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook calc_02:

| Notebook | Sections |
|---|---|
| plot_03a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| plot_03b | 3. Quantile response; 4. Change in width, and which tail |
| plot_03c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| plot_03d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`EXAMPLE_MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `EXAMPLE_MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 5. The whole distribution | where the mean and the width changed (after Bracegirdle et al., 2024) | sections 1 and 4 |
| 6. Additivity | historical against the sum of the single forcings | none |
| 7. Zonal means and the sea-ice edge | sections 1 and 4 by latitude (after Bracegirdle et al., 2024); the change in sea ice, and how it moves with the temperature year to year | sections 1 and 4, zonally averaged |

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 5 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook calc_04): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook plot_05. The cells before each figure pick out its data and show them. Each figure is then drawn layer by layer with `extant/plots/antarctica.py`: `map_figure_size` and `map_axes` make the figure, its maps and colour bars, then each layer (a field, its dots, each ice edge) is drawn on those axes by its own call, a row of a different kind being a slice of them (`axes[2]`).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook calc_02. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
import xarray as xr

from plotting_modules import formatting
from plotting_modules.constants import FORCING_COLORS, SEASONS
from extant import config, preprocessing as pp, response_change as rc, sea_ice
from extant.config import ALPHA, FORCED_EXPERIMENTS, LAST_FULL_YEAR, WINDOW
from extant import significance as sig, zonal
from extant.logger import setup_logging
from extant.plots import antarctica, charts, forced_response as fr, response_change as rcp, zonal as zp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `LAST_FULL_YEAR`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The model and experiment of the "what goes in" and "one model" figures
EXAMPLE_MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']


In [ ]:
#(t): Where notebook calc_02 (and calc_04) saved the results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks plot_04 and plot_05), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook plot_05 (*Check the Colour Ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): The sea ice of 7.7 and 7.8, as in notebook plot_02: the concentration, its change (blue: more ice, red: less, so ice
#(t): lost reads like warming), and its correlation with the temperature
concentration_scale = fr.Scale(np.arange(0, 101, 10), 'Blues', 'Sea-ice concentration (%)', extend='neither')
concentration_change_scale = fr.Scale(np.arange(-55, 56, 10), 'RdBu', 'Difference in sea-ice concentration (%)',
                                      ticks=np.arange(-50, 51, 10))
correlation_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 1), 'PuOr_r', 'Correlation of sea ice and temperature',
                             extend='neither')

In [ ]:
#(t): The legend entries of the maps: the dots, and the two ice edges (hist-nat's dashed, the experiment's solid)
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                                    label=f'Not significant (p ≥ {ALPHA:g})')
hist_nat_edge_handle = plt.Line2D([], [], color='k', linestyle='dashed', label="hist-nat's ice edge")
experiment_edge_handle = plt.Line2D([], [], color='k', label="The experiment's ice edge")

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the Data and the Saved Results

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (the seasonal means, used only for the figures at one grid point in 5.5) and `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebook calc_02's and notebook calc_04's results are opened from.

### Sample data

In [ ]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]
sample_names

In [ ]:
#(t): Each file of config.MAIN_EXPERIMENTS opened, as a DataTree /<model>/<experiment>
lesfmip_sample_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                             for path, (model, experiment) in zip(sample_files, sample_names)
                                             if experiment in config.MAIN_EXPERIMENTS})

In [ ]:
#(t): Seasonal means in analysis units, through the same pre-processing as notebook calc_01
lesfmip_season_tree = pp.seasonal_tree(lesfmip_sample_tree, VARIABLE.name).compute()

In [ ]:
#(t): The sample's results: notebook calc_02's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
RESULTS_DIR, ICE_RESULTS_DIR

### Full data

In [ ]:
#(t): The seasonal means notebook calc_01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
seasonal_path

In [ ]:
#(t): Opened lazily
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
#(t): The full data's results: notebook calc_02's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
RESULTS_DIR, ICE_RESULTS_DIR

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)

### Saved results

Everything sections 5, 6 and 7 use, from `RESULTS_DIR`, one store per cell, loaded into memory.

In [ ]:
#(t): The hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()

In [ ]:
#(t): hist-nat's final-years quantiles
hist_nat_final_q_da = xr.open_zarr(RESULTS_DIR / 'hist_nat_final_quantiles.zarr')[VARIABLE.name].load()

In [ ]:
#(t): Every experiment's final-years quantiles
final_q_da = xr.open_zarr(RESULTS_DIR / 'final_quantiles.zarr')[VARIABLE.name].load()

In [ ]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()

In [ ]:
#(t): The change in each tail, Q95 − Q50 and Q50 − Q05
tails_ds = xr.open_zarr(RESULTS_DIR / 'tails.zarr').load()

In [ ]:
#(t): S/N
lesfmip_seasonal_sn_ds = xr.open_zarr(RESULTS_DIR / 'sn.zarr').load()

In [ ]:
#(t): Each experiment's change from its own 1850-1900 baseline
own_changes_da = xr.open_zarr(RESULTS_DIR / 'own_changes.zarr')['own_change'].load()

In [ ]:
#(t): The zonal-mean test of the mean
zonal_mean_ds = xr.open_zarr(RESULTS_DIR / 'zonal_mean_test.zarr').load()

In [ ]:
#(t): The zonal-mean test of the width
zonal_width_ds = xr.open_zarr(RESULTS_DIR / 'zonal_width_test.zarr').load()

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = mean_bootstrap_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps (sections 5 and 7) and the zonal means (section 7), and the sea ice of 7.7 and 7.8, come from notebook calc_04. Run **one** of the next two sections. Both leave `ice_extents_tree` (each member's seasonal extent), `ice_climatology_da` (the ensemble-mean concentration over the final years), `ice_edges_da` (the equivalent latitude of the edge in the final years), and `model_ice_da` and `median_ice_da` (`EXAMPLE_MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out. *With sea ice* also opens the change in concentration 7.7 uses; without it, 7.6–7.8 cannot run.

#### Without sea ice

Before notebook calc_04 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_extents_tree = ice_climatology_da = ice_edges_da = model_ice_da = median_ice_da = None

#### With sea ice

In [ ]:
#(t): Each member's seasonal extent and its model's continent, the ensemble-mean concentration over the final years,
#(t): the equivalent latitude of each model's edge, and EXAMPLE_MODEL's concentration and the models' median (for the
#(t): edges)
ice_extents_tree = xr.open_datatree(ICE_RESULTS_DIR / 'extent.zarr', engine='zarr').load()
ice_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'climatology.zarr')['siconc'].load()
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)
model_ice_da = ice_climatology_da.sel(model=EXAMPLE_MODEL)
median_ice_da = ice_climatology_da.median('model')

In [ ]:
#(t): Each experiment's change in concentration against hist-nat, every model and season (%), for 7.7
ice_change_da = xr.open_zarr(ICE_RESULTS_DIR / 'ice_change.zarr')['siconc'].load()

## 5. Whole Distribution: Where the Mean and the Width Changed

Sections 1, 3 and 4 together, in the form of Bracegirdle et al. (2024, Fig. 1): the change in the low end of the distribution (Q05), its middle (the mean), its high end (Q95) and its width (Q95 − Q05). Equal changes in Q05, the mean and Q95 are a pure shift; a bigger change in Q05 than in Q95 is a distribution narrowing from below. (The values behind these changes are in 1.1, 3.1 and 4.1, in notebooks plot_03a and plot_03b.)

Then each grid point is put in one of nine classes: the mean significantly down, unchanged or up (section 1's test), times the width significantly narrower, unchanged or wider (section 4's test). The 3 × 3 key reads like a table: red is a warmer mean, blue a cooler one; yellow mixed in is a narrower distribution, purple a wider one.

`MEAN_TEST` chooses the test for the mean: `'significant'` is the hist-nat bootstrap, as in section 1; `'robust'` asks for $|\mathrm{S/N}| \ge 2$ as well (section 2, notebook plot_03a), and `'emerged'` for that alone.

In [ ]:
#(t): Which test says the mean has changed: 'significant' (hist-nat bootstrap, section 1), 'emerged' (S/N) or 'robust' (both)
MEAN_TEST = 'significant'

In [ ]:
#(t): The year of section 2 (notebook plot_03a): the centre of the final WINDOW years
sn_year = int(lesfmip_seasonal_sn_ds['year'][LAST_FULL_YEAR])

In [ ]:
#(t): S/N in the year of section 2, for the 'emerged' and 'robust' tests
sn_final_da = lesfmip_seasonal_sn_ds[VARIABLE.name].sel(year=sn_year)

In [ ]:
#(t): Every change and its test in one Dataset on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'], sn_final_da, qrange_change_ds,
                              tails_ds, alpha=ALPHA)

In [ ]:
#(t): hist-nat's distribution over the final WINDOW years: Q05, the mean and Q95, and the width of its internal
#(t): variability (as tested in section 4)
hist_nat_fields_ds = xr.Dataset({
    'Q05': hist_nat_final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['hist_nat_mean'],
    'Q95': hist_nat_final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['hist_nat_qrange'],
})

In [ ]:
#(t): Every experiment's, over its final WINDOW years
experiment_fields_ds = xr.Dataset({
    'Q05': final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['experiment_mean'],
    'Q95': final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['experiment_qrange'],
})

In [ ]:
#(t): The change in each: the same as sections 1, 3 and 4
change_fields_ds = experiment_fields_ds - hist_nat_fields_ds

In [ ]:
#(t): Which changes are significant: the mean by MEAN_TEST, the width by the hist-nat bootstrap (Q05 and Q95 have no test)
change_significant_ds = xr.Dataset({'Mean': change_ds[f'mean_{MEAN_TEST}'], 'Q95 − Q05': change_ds['width_significant']})

In [ ]:
#(t): Not significant, where there is data
change_not_significant_ds = ~change_significant_ds & change_fields_ds[list(change_significant_ds)].notnull()

In [ ]:
#(t): The sign of each significant change: +1 up, -1 down, 0 not significant
mean_sign_da = sig.significant_sign(change_fields_ds['Mean'], change_significant_ds['Mean'])
width_sign_da = sig.significant_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): The joint class, 0 to 8, in the order of the 3 x 3 key
joint_class_da = rc.joint_class(mean_sign_da, width_sign_da)

### 5.1 What goes in

The change in each part of the distribution, `EXPERIMENT` against hist-nat for `EXAMPLE_MODEL`, dotted where the mean or the width change is not significant; then the class the two give. The three changes of the distribution share one colour scale; the width has its own.

In [ ]:
#(t): EXAMPLE_MODEL and EXPERIMENT, every season: the change
model_change_fields_ds = change_fields_ds.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)
model_change_fields_ds

In [ ]:
#(t): Where the change is not significant
model_change_not_significant_ds = change_not_significant_ds.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                                season=SEASONS)
model_change_not_significant_ds

In [ ]:
#(t): The joint class
model_joint_class_da = joint_class_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)
model_joint_class_da

In [ ]:
#(t): What goes in: the change in Q05, the mean and Q95 (one colour bar), in the width, and what changed (its key),
#(t): each with both ice edges; the mean and the width dotted where not significant
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=5, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1,
                                                    n_keys=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=5, n_cols=len(SEASONS))
#(c): The change's colour bar under rows 1-3, the width's under row 4, the key under row 5
change_cax = antarctica.colorbar_axis_below(axes[:3])
width_cax = antarctica.colorbar_axis_below(axes[3])
key_ax = antarctica.key_axis_below(axes[4])

#(c): Rows 1-3: the change in Q05, the mean and Q95, on one colour bar
antarctica.heatmaps_of_antarctica(model_change_fields_ds['Q05'], None, 'season', axes=axes[0], cax=change_cax,
                                  **scales.change.colours())
antarctica.heatmaps_of_antarctica(model_change_fields_ds['Mean'], None, 'season', axes=axes[1], **scales.change.colours())
antarctica.not_significant_dots_grid(model_change_not_significant_ds['Mean'], None, 'season', axes=axes[1])
antarctica.heatmaps_of_antarctica(model_change_fields_ds['Q95'], None, 'season', axes=axes[2], **scales.change.colours())
#(c): Row 4: the change in the width
antarctica.heatmaps_of_antarctica(model_change_fields_ds['Q95 − Q05'], None, 'season', axes=axes[3], cax=width_cax,
                                  **scales.spread_change.colours())
antarctica.not_significant_dots_grid(model_change_not_significant_ds['Q95 − Q05'], None, 'season', axes=axes[3])
#(c): Row 5: what changed, the mean and the width together, with its key instead of a colour bar
antarctica.heatmaps_of_antarctica(model_joint_class_da, None, 'season', axes=axes[4], **scales.joint.colours())
antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)
#(c): Both ice edges on every row, if the sea ice was opened
if model_ice_da is not None:
    for row_axes in axes:
        antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season', axes=row_axes,
                             linestyle='dashed')
        antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), None, 'season', axes=row_axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)', 'What changed'])
antarctica.tag_panels(axes)
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f"The distribution's change: {EXAMPLE_MODEL}, {EXPERIMENT}, final {WINDOW} years against hist-nat");

### 5.2 One model

What changed, for every experiment of `EXAMPLE_MODEL` (1.2 and 4.2, in notebooks plot_03a and plot_03b, show the mean and the width changes themselves).

In [ ]:
#(t): EXAMPLE_MODEL's joint class, every experiment and season
model_forcing_class_da = joint_class_da.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS, season=SEASONS)
model_forcing_class_da

In [ ]:
#(t): EXAMPLE_MODEL's joint class in every experiment (rows), with hist-nat's ice edge (dashed) and each experiment's
#(t): (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=0,
                                                    n_keys=1, n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
key_ax = antarctica.key_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_forcing_class_da, 'experiment', 'season', axes=axes, **scales.joint.colours())
antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)
#(c): The ice edges, if the sea ice was opened: hist-nat's on every row (given the rows' experiments), each experiment's
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                          drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                         'experiment', 'season', axes=axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                         axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: where the mean and the width have changed, final {WINDOW} years against hist-nat');

### 5.3 Every model

The joint class of every model; one figure per experiment.

In [ ]:
#(t): The joint class for every model, experiment and season
all_class_da = joint_class_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
all_class_da

In [ ]:
#(t): Every model's joint class (rows), one figure per experiment, with each model's ice edges
models = all_class_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=0,
                                                        n_keys=1, n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    key_ax = antarctica.key_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_class_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      **scales.joint.colours())
    antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)
    #(c): The ice edges in the rows' order (empty for a model without sea ice), if the sea ice was opened
    if ice_climatology_da is not None:
        rows_ice_da = ice_climatology_da.reindex(model=models).sel(season=SEASONS)
        antarctica.ice_edges(rows_ice_da.sel(experiment='hist-nat'), 'model', 'season', axes=axes, linestyle='dashed')
        antarctica.ice_edges(rows_ice_da.sel(experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'Where the mean and the width have changed, every model: {experiment}')

### 5.4 How many models

The multi-model median of the mean change and of the width change (the top two rows), then the number of models where the mean has changed and the distribution has widened, or narrowed. One figure per experiment.

In [ ]:
#(t): The mean change for every model, experiment and season (as in section 1, notebook plot_03a)
all_mean_change_da = mean_bootstrap_ds['mean_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Its multi-model median: the top row of the figures below
median_mean_change_da = all_mean_change_da.median('model')
median_mean_change_da

In [ ]:
#(t): The change in width for every model, experiment and season (as in section 4, notebook plot_03b)
all_width_change_da = qrange_change_ds['qrange_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Its multi-model median: the second row
median_width_change_da = all_width_change_da.median('model')
median_width_change_da

In [ ]:
#(t): How many models changed in the mean and became wider, and how many narrower
joint_counts_da = rc.joint_change_counts(change_ds, mean_test=MEAN_TEST)

In [ ]:
#(t): Every experiment and season
all_joint_counts_da = joint_counts_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): The counts of 'mean and narrower' as negative numbers, so that one colour bar (blues for narrower, reds for
#(t): wider) serves both rows of counts
signed_joint_counts_da = xr.concat([all_joint_counts_da.sel(change=['mean and wider']),
                                    -all_joint_counts_da.sel(change=['mean and narrower'])], dim='change')
signed_joint_counts_da

In [ ]:
#(t): Their colours: blues and reds up to half the models, purples and browns beyond, white for none
joint_count_colours = fr.count_scale(N_MODELS, 'Number of models where the mean changed and the distribution '
                                               'narrowed or widened').colours()

In [ ]:
#(t): The multi-model median of the mean change and of the width change (top rows), then how many models changed in
#(t): both, one figure per experiment, with the ice edges of the models' median concentration
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbar_rows=3, n_legends=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS))
    #(c): Each colour bar under the rows it is for: the mean's under row 1, the width's under row 2, the counts' under
    #(c): rows 3 and 4
    mean_cax = antarctica.colorbar_axis_below(axes[0])
    width_cax = antarctica.colorbar_axis_below(axes[1])
    count_cax = antarctica.colorbar_axis_below(axes[2:])

    antarctica.heatmaps_of_antarctica(median_mean_change_da.sel(experiment=experiment), None, 'season', axes=axes[0],
                                      cax=mean_cax, **scales.change.colours())
    antarctica.heatmaps_of_antarctica(median_width_change_da.sel(experiment=experiment), None, 'season', axes=axes[1],
                                      cax=width_cax, **scales.spread_change.colours())
    antarctica.heatmaps_of_antarctica(signed_joint_counts_da.sel(experiment=experiment), 'change', 'season',
                                      axes=axes[2:], cax=count_cax, **joint_count_colours)
    #(c): Both ice edges on every row, if the sea ice was opened
    if median_ice_da is not None:
        for row_axes in axes:
            antarctica.ice_edges(median_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season',
                                 axes=row_axes, linestyle='dashed')
            antarctica.ice_edges(median_ice_da.sel(experiment=experiment, season=SEASONS), None, 'season',
                                 axes=row_axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Median\nmean change', 'Median\nwidth change', 'Mean and\nwider', 'Mean and\nnarrower'])
    fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'How many models have changed in both the mean and the width: {experiment}')

### 5.5 A closer look at one point

At a single grid point and season, the final years of each experiment (members pooled) against hist-nat's:

- **a)** the two distributions, with arrows from each hist-nat quantile (Q05, Q50, Q95) to the experiment's. Equal arrows are a pure shift;
- **b)** the change in every quantile, for every experiment, with a bootstrap (whole members) 5–95% range. A flat curve is a pure shift; a rising one, the upper tail moving more; a falling one, the lower tail.

The point and season are where the mean and the width both changed and the width changed most (south of 60°S, any season). Set `change_point` to look anywhere else.

In [ ]:
#(t): The point and its season: where the mean and the width both changed, and the width changed most
change_point = rc.strongest_joint_change(change_ds, EXAMPLE_MODEL, EXPERIMENT, mean_test=MEAN_TEST)
# change_point = dict(season='JJA', lat=-75.0, lon=-180.0)   # or choose one

In [ ]:
#(t): The experiments EXAMPLE_MODEL has, hist-nat first, then in the order of the figures
point_experiments = [e for e in ['hist-nat', *FORCED_EXPERIMENTS] if e in lesfmip_season_tree[EXAMPLE_MODEL].children]

In [ ]:
#(t): Their final years at that point and season, on (member, year)
point_samples = rc.final_years(lesfmip_season_tree, EXAMPLE_MODEL, point_experiments, change_point, years=WINDOW,
                               variable=VARIABLE.name)
point_samples[EXPERIMENT]

In [ ]:
#(t): The change in every quantile of each experiment against hist-nat, with its bootstrap range
point_shifts = {e: rc.quantile_shift(point_samples[e], point_samples['hist-nat']) for e in point_experiments if e != 'hist-nat'}
point_shifts[EXPERIMENT]

In [ ]:
#(t): How the distribution changes at change_point: a) hist-nat's and EXPERIMENT's final years (members pooled), with
#(t): arrows from each hist-nat quantile to the experiment's; b) the change in every quantile, every experiment, with
#(t): its bootstrap 5-95% range
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=2, n_legends=0, panel_width=5.2, panel_height=3.4),
                 layout='constrained')
axes = fig.add_gridspec(1, 2).subplots(squeeze=False)
rcp.plot_point_distributions(point_samples, point_shifts[EXPERIMENT], EXPERIMENT,
                             test_ds=change_ds.sel(model=EXAMPLE_MODEL,
                                                   experiment=EXPERIMENT).sel(change_point), units=VARIABLE.units,
                             ax=axes[0, 0])
rcp.plot_quantile_shifts(point_shifts, EXPERIMENT, ax=axes[0, 1])
axes[0, 0].set_title(f'a) hist-nat vs {EXPERIMENT}: final years, members pooled', loc='left')
axes[0, 0].set_xlabel(f'Seasonal-mean tas ({VARIABLE.units})')
axes[0, 0].set_ylabel('Density')
axes[0, 1].set_title('b) Change in every quantile (bootstrap 5–95%)', loc='left')
axes[0, 1].set_xlabel('Quantile (%): flat, a pure shift; rising, the warm tail warms faster')
axes[0, 1].set_ylabel(f'Change vs hist-nat ({VARIABLE.units})')
fig.suptitle(f'How the distribution changes: {EXAMPLE_MODEL}, {change_point["season"]} at '
             f'{formatting.format_latlon(change_point)}', x=0.01, ha='left');

### 5.6 Every model and forcing, as area means

Area means south of 60°S. First every model's change in the mean and in the width for each forcing (dots: one per model; bars: the multi-model median), then the two against each other, then the lower tail's change against the upper tail's (above the dashed 1:1 line the upper tail stretched more).

In [ ]:
#(t): Area means south of 60°S of every change, and the share of the area in each class
change_regional_ds = rc.regional_mean(change_ds.sel(experiment=FORCED_EXPERIMENTS), lat_max=-60, mean_test=MEAN_TEST)

In [ ]:
#(t): Area means south of 60°S: every model's change (markers) and the multi-model median (bars) for each forcing; the
#(t): mean change in the top row, the change in width below, seasons across
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=2, n_cols=len(SEASONS), n_legends=1, panel_height=2.0),
                 layout='constrained')
axes = fig.add_gridspec(2, len(SEASONS)).subplots(sharex=True, sharey='row', squeeze=False)
charts.model_dots_grid(change_regional_ds['mean_change'].sel(season=SEASONS), 'experiment', None, 'season',
                       axes=axes[0], color=FORCING_COLORS)
charts.model_dots_grid(change_regional_ds['width_change'].sel(season=SEASONS), 'experiment', None, 'season',
                       axes=axes[1], color=FORCING_COLORS)
axes[0, 0].set_ylabel(f'Mean change ({VARIABLE.units})')
axes[1, 0].set_ylabel(f'Change in Q95 − Q05 ({VARIABLE.units})')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=charts.model_legend_handles(change_regional_ds['model'].values)
           + [Patch(color='0.6', alpha=0.5, label='Multi-model median')],
           loc='outside lower center', ncols=change_regional_ds.sizes['model'] + 1, frameon=False)
fig.suptitle('Area mean south of 60°S: every model (markers) and the multi-model median (bars)', x=0.01, ha='left');

In [ ]:
#(t): Area means south of 60°S: the change in width against the mean change, one point per model (marker) and
#(t): forcing (colour), one panel per season; each corner says what a point there means
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(change_regional_ds['mean_change'].sel(season=season),
                                   change_regional_ds['width_change'].sel(season=season), ax=ax)
    rcp.plot_quadrant_labels('mean_change', 'width_change', ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel(rcp.SCATTER_LABELS['mean_change'][0])
axes[0, 0].set_ylabel(rcp.SCATTER_LABELS['width_change'][0])
legend_axes[0].legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS), loc='center',
                      ncols=len(FORCED_EXPERIMENTS),
                      frameon=False)
legend_axes[1].legend(handles=charts.model_legend_handles(change_regional_ds['model'].values), loc='center',
                      ncols=change_regional_ds.sizes['model'], frameon=False)
fig.suptitle('Area mean south of 60°S: change in width against mean change', x=0.01, ha='left');

In [ ]:
#(t): ... the upper-tail change against the lower-tail change: above the dashed 1:1 line the upper tail stretched more
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(change_regional_ds['lower_tail_change'].sel(season=season),
                                   change_regional_ds['upper_tail_change'].sel(season=season), ax=ax)
    rcp.plot_quadrant_labels('lower_tail_change', 'upper_tail_change', ax=ax)
    ax.axline((0, 0), slope=1, color='0.55', linewidth=0.9, linestyle=(0, (4, 2)), zorder=0)
    ax.set_title(season, loc='left')
    ax.set_xlabel(rcp.SCATTER_LABELS['lower_tail_change'][0])
axes[0, 0].set_ylabel(rcp.SCATTER_LABELS['upper_tail_change'][0])
legend_axes[0].legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS), loc='center',
                      ncols=len(FORCED_EXPERIMENTS),
                      frameon=False)
legend_axes[1].legend(handles=charts.model_legend_handles(change_regional_ds['model'].values), loc='center',
                      ncols=change_regional_ds.sizes['model'], frameon=False)
fig.suptitle('Area mean south of 60°S: upper-tail against lower-tail change', x=0.01, ha='left');

In [ ]:
#(t): The share of the area south of 60°S where neither, the mean only, the width only, or both changed (%)
class_columns = [f'fraction_{name.replace(" ", "_")}' for name in rc.CHANGE_CLASSES.values()]
class_share_df = (100 * change_regional_ds[class_columns].sel(season=SEASONS)).to_dataframe().round(0)
class_share_df

## 6. Additivity: Does historical Equal the Sum of the Single Forcings?

Each experiment's change from its own pre-industrial baseline: its final 21 years minus its own 1850–1900 (notebook calc_02, section 6). A single-forcing experiment contains *only* its forcing, so this is its response; measured from hist-nat, as in the other sections, a sum of four forcings would subtract hist-nat's volcanic cooling four times.

$$
\Delta_e = \bar{x}^{\,e}_{\text{final } L \text{ years}} - \bar{x}^{\,e}_{1850\text{–}1900},
\qquad
\text{residual} = \Delta_{\mathrm{historical}} - \left(\Delta_{\mathrm{nat}} + \Delta_{\mathrm{GHG}} + \Delta_{\mathrm{aer}} + \Delta_{\mathrm{O_3}}\right).
$$

The residual holds everything the parts leave out: land use (hist-lu, which should be very small over Antarctica), any other forcing, and non-additive interaction between forcings. There is no test, so the last part is a table rather than a count.

In [ ]:
#(t): Every term: each part, their sum, historical and the residual; and each part's fraction of historical
additivity_ds = rc.additivity(own_changes_da)

### 6.1 What goes in

Each single forcing's change from its own baseline; `EXAMPLE_MODEL`.

In [ ]:
#(t): EXAMPLE_MODEL's single forcings, every season
model_parts_da = additivity_ds['response'].sel(model=EXAMPLE_MODEL, season=SEASONS, term=list(rc.ADDITIVE_PARTS))
model_parts_da

In [ ]:
#(t): Each term (rows), every season
terms = model_parts_da['term'].values
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(terms), n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(terms), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_parts_da, 'term', 'season', axes=axes, cax=cax, **scales.change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, terms)
fig.suptitle(f'What goes into the additivity: {EXAMPLE_MODEL}, final {WINDOW} years minus 1850–1900');

### 6.2 One model

The sum of the parts, historical, and the residual for `EXAMPLE_MODEL`.

In [ ]:
#(t): EXAMPLE_MODEL's sum of parts, historical and residual, every season
model_additivity_da = additivity_ds['response'].sel(model=EXAMPLE_MODEL, season=SEASONS,
                                                    term=['sum of parts', 'historical', 'residual'])
model_additivity_da

In [ ]:
#(t): Each term (rows), every season
terms = model_additivity_da['term'].values
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(terms), n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(terms), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_additivity_da, 'term', 'season', axes=axes, cax=cax, **scales.change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, terms)
fig.suptitle(f'{EXAMPLE_MODEL}: historical against the sum of the single forcings');

### 6.3 Every model

One figure each for the sum of the parts, historical, and the residual.

In [ ]:
#(t): Every model's sum of parts, historical and residual, every season
all_additivity_da = additivity_ds['response'].sel(season=SEASONS, term=['sum of parts', 'historical', 'residual'])
all_additivity_da

In [ ]:
#(t): Every model (rows), one figure per term
models = all_additivity_da['model'].values
for term in all_additivity_da['term'].values:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_additivity_da.sel(term=term), 'model', 'season', axes=axes, cax=cax,
                                      **scales.change.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.suptitle(f'Historical against the sum of the single forcings, every model: {term}')

### 6.4 Every model, as numbers

Area means south of 60°S of every term, and each part's share of historical's area mean (the share of the area means, which does not blow up where historical's change is small at a point).

In [ ]:
#(t): Area means south of 60°S of every term, every season
antarctic_terms_da = sig.area_mean(additivity_ds['response'].sel(season=SEASONS, lat=slice(None, -60)))

In [ ]:
#(t): As a table: models and seasons down, terms across
antarctic_terms_df = antarctic_terms_da.to_series().unstack('term')[list(additivity_ds['term'].values)].round(2)

In [ ]:
#(t): Each part's and the residual's share of historical (%)
share_columns = [*rc.ADDITIVE_PARTS, 'residual']
share_of_historical_df = (100 * antarctic_terms_df[share_columns].div(antarctic_terms_df['historical'], axis=0)).round(0)
share_of_historical_df

## 7. Zonal Means and the Sea-Ice Edge

After Bracegirdle et al. (2024, *npj Clim. Atmos. Sci.* 7, 276; [doi:10.1038/s41612-024-00822-y](https://doi.org/10.1038/s41612-024-00822-y)), Figs 1–3, 6 and 8, which compare the change in extreme seasons with the change in the background climate, latitude by latitude, and relate the difference to the retreating sea ice. Their methods differ from ours, so these figures keep their layout but show this project's changes:

| Column | Here | The paper |
|---|---|---|
| Mean | $\Delta$ mean (section 1) | the background climate (smoothed ensemble mean) |
| Low extremes | $\Delta(Q_{05} - Q_{50})$ of the internal variability: > 0 where the low extremes move up towards the middle | p10, minus the background |
| High extremes | $\Delta(Q_{95} - Q_{50})$: > 0 where the high extremes move away from the middle | p90, minus the background |
| Width | $\Delta(Q_{95} - Q_{05})$ = high − low (section 4) | p90 − p10 |

**Every model is shown on its own** (the paper averages five). The zonal means are tested (notebook calc_02, section 8): dots on a line mark a significant change in the mean or the width (both by the hist-nat bootstrap), and the grey band is what hist-nat's own variability gives the width. The dashed lines are the sea-ice edge (equivalent latitude, notebook calc_04 section 3) in each experiment's final years. These four columns are the paper's, so here the seasons run down the page (7.1, 7.2), or get a figure each (7.3, 7.4).

The paper's key result for temperature: in winter (JJA), over the retreating sea ice, the cold extremes warm much more than the warm extremes, so the distribution narrows. Here that would show as a positive low-extreme change and a negative width change just poleward of the ice edge, in the experiments whose edge has moved.

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)

### 7.1 What goes in

The maps behind the zonal means for `EXAMPLE_MODEL` and `EXPERIMENT` (the paper's Figs 1 and 3), seasons down, with the 15% ice edge of the experiment (solid) and of hist-nat (dashed). Dotted where the mean or the width change is not significant.

In [ ]:
#(t): EXAMPLE_MODEL and EXPERIMENT's changes: the mean, the low and high extremes and the width, as Bracegirdle et al.
#(t): (2024)
#(t): split them (rc.extreme_changes: the extremes measured from the median)
zonal_model_change_ds = change_ds.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)
zonal_model_extremes_ds = rc.extreme_changes(zonal_model_change_ds)
zonal_model_extremes_ds

In [ ]:
#(t): Not significant: the mean by MEAN_TEST, the width by the hist-nat bootstrap (the extremes have no test)
zonal_mean_not_significant_da = ~zonal_model_change_ds[f'mean_{MEAN_TEST}'] & zonal_model_change_ds['mean_change'].notnull()
zonal_width_not_significant_da = ~zonal_model_change_ds['width_significant'] & zonal_model_change_ds['width_change'].notnull()
zonal_width_not_significant_da

In [ ]:
#(t): The maps behind the zonal means (the paper's Figs 1 and 3): the four changes across, the seasons down; the mean
#(t): on its own colour bar, the extremes and the width on another; with both ice edges
extreme_columns = list(rc.EXTREMES)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(SEASONS), n_cols=len(extreme_columns),
                                                    n_colorbar_rows=1, n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(SEASONS), n_cols=len(extreme_columns))
#(c): Side by side under the last row: the mean's colour bar under column 1, the extremes' under columns 2-4
mean_cax = antarctica.colorbar_axis_below(axes[:, 0])
extremes_cax = antarctica.colorbar_axis_below(axes[:, 1:])

#(c): Column 1: the mean change, dotted where not significant
mean_cbar = antarctica.heatmaps_of_antarctica(zonal_model_extremes_ds['mean_change'], 'season', None, axes=axes[:, 0],
                                              cax=mean_cax, **scales.change.colours())
#(c): Every other tick, so they fit under one column
mean_cbar.set_ticks(scales.change.levels[::2])
antarctica.not_significant_dots_grid(zonal_mean_not_significant_da, 'season', None, axes=axes[:, 0])
#(c): Columns 2-4: the low and high extremes and the width, on one colour bar; the width dotted where not significant
extremes_colours = dict(scales.spread_change.colours(), label=f'Extremes and width ({VARIABLE.units})')
antarctica.heatmaps_of_antarctica(zonal_model_extremes_ds[extreme_columns[1:]].to_dataarray('quantity'), 'season', 'quantity',
                                  axes=axes[:, 1:], cax=extremes_cax, **extremes_colours)
antarctica.not_significant_dots_grid(zonal_width_not_significant_da, 'season', None, axes=axes[:, 3])
#(c): Both ice edges on every map, column by column, if the sea ice was opened
if model_ice_da is not None:
    for column_axes in axes.T:
        antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), 'season', None, axes=column_axes,
                             linestyle='dashed')
        antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), 'season', None, axes=column_axes)

antarctica.label_columns(axes, [f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}' for column in extreme_columns])
antarctica.label_rows(axes, SEASONS)
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f'What goes into the zonal means: {EXAMPLE_MODEL}, {EXPERIMENT}');

### 7.2 One model

`EXAMPLE_MODEL`'s change at each latitude, every experiment as a line, seasons down.

In [ ]:
#(t): EXAMPLE_MODEL's four columns, every season, on one 'column' dim
model_zonal_da = zonal_ds[['mean_change', 'low_extreme_change', 'high_extreme_change', 'width_change']].sel(
    model=EXAMPLE_MODEL, season=SEASONS).to_dataarray('column')
model_zonal_da

In [ ]:
#(t): Quick look: against latitude, one line per experiment; the four columns down, the seasons across
model_zonal_da.plot.line(x='lat', hue='experiment', row='column', col='season', sharey='row');

In [ ]:
#(t): The zonal figures' four columns, and the legend entries they share: a dot where a change is significant,
#(t): hist-nat's own band of the width change, and the ice edge
zonal_columns = list(rc.EXTREMES)
zonal_legend_handles = [
    Line2D([], [], linestyle='none', marker='o', markersize=5, color='k', markeredgecolor='white',
           label=f'Significant (p < {ALPHA:g})'),
    Patch(color=zp.NULL_COLOR, label='hist-nat variability alone (95%)'),
    Line2D([], [], color='k', linewidth=1.0, linestyle=zp.EDGE_LINESTYLE, label='Sea-ice edge (equivalent latitude)'),
]

In [ ]:
#(t): EXAMPLE_MODEL's zonal-mean change, seasons down and the four changes across: one line per experiment, a dot where
#(t): the
#(t): mean's or the width's change is significant, hist-nat's own 95% of the width change grey, and each experiment's
#(t): ice edge (and hist-nat's) dashed. The extremes and the width share one scale
model_zonal_ds = zonal_ds.sel(model=EXAMPLE_MODEL, season=SEASONS)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(SEASONS), n_cols=len(zonal_columns), n_legends=1,
                                                  panel_height=1.95), layout='constrained')
axes = fig.add_gridspec(len(SEASONS), len(zonal_columns)).subplots(sharex=True, sharey='col', squeeze=False)
for ax in axes.ravel():
    ax.axhline(0, color='0.64', linewidth=0.8, zorder=1)
charts.bands_grid(model_zonal_ds['null_lower'], model_zonal_ds['null_upper'], 'lat', 'season', None, axes=axes[:, 3],
                  color=zp.NULL_COLOR)
for experiment in FORCED_EXPERIMENTS:
    experiment_zonal_ds = model_zonal_ds.sel(experiment=experiment)
    charts.lines_grid(experiment_zonal_ds[zonal_columns].to_dataarray('column'), 'lat', 'season', 'column', axes=axes,
                      color=FORCING_COLORS[experiment], linewidth=1.6)
    charts.significant_dots_grid(experiment_zonal_ds['mean_change'], experiment_zonal_ds['mean_pvalue'] < ALPHA, 'lat',
                                 'season', None, axes=axes[:, 0], color=FORCING_COLORS[experiment])
    charts.significant_dots_grid(experiment_zonal_ds['width_change'], experiment_zonal_ds['width_pvalue'] < ALPHA,
                                 'lat', 'season', None, axes=axes[:, 3], color=FORCING_COLORS[experiment])
#(c): Each experiment's edge on all four columns: given a column dim with the four, so each column finds it
if ice_edges_da is not None:
    for experiment in ['hist-nat', *FORCED_EXPERIMENTS]:
        zp.ice_edge_lines_grid(ice_edges_da.sel(model=EXAMPLE_MODEL, experiment=experiment, season=SEASONS).expand_dims(
            column=zonal_columns), 'season', 'column', axes=axes, color=FORCING_COLORS[experiment])

for ax, column in zip(axes[0], zonal_columns):
    ax.set_title(f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}', loc='left')
for ax, season in zip(axes[:, 0], SEASONS):
    ax.set_ylabel(f'{season}\n({VARIABLE.units})')
for ax in axes[-1]:
    ax.set_xlabel('Latitude')
#(c): The x axis is shared: its labels and limits set once
axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
#(c): The extremes and the width on one scale, so the width is visibly the high column minus the low
extremes_ylim = (min(ax.get_ylim()[0] for ax in axes[0, 1:]), max(ax.get_ylim()[1] for ax in axes[0, 1:]))
for ax in axes[0, 1:]:
    ax.set_ylim(extremes_ylim)
fig.legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS) + zonal_legend_handles, loc='outside lower center',
           ncols=len(FORCED_EXPERIMENTS) + len(zonal_legend_handles), frameon=False)
fig.suptitle(f'{EXAMPLE_MODEL}: zonal-mean change, final {WINDOW} years against hist-nat', x=0.01, ha='left');

### 7.3 Every model

One row per model, the experiments as lines, one figure per season. The mean has its own scale; the extremes and the width share one, so the width column is the high column minus the low.

In [ ]:
#(t): Every model's zonal-mean change, one figure per season: models down, the four changes across, one line per
#(t): experiment; dots, band and edges as in 7.2
zonal_models = list(zonal_ds['model'].values)
for season in SEASONS:
    season_zonal_ds = zonal_ds.sel(season=season)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(zonal_models), n_cols=len(zonal_columns), n_legends=1,
                                                      panel_height=1.95), layout='constrained')
    axes = fig.add_gridspec(len(zonal_models), len(zonal_columns)).subplots(sharex=True, sharey='col', squeeze=False)
    for ax in axes.ravel():
        ax.axhline(0, color='0.64', linewidth=0.8, zorder=1)
    charts.bands_grid(season_zonal_ds['null_lower'], season_zonal_ds['null_upper'], 'lat', 'model', None,
                      axes=axes[:, 3], color=zp.NULL_COLOR)
    for experiment in FORCED_EXPERIMENTS:
        experiment_zonal_ds = season_zonal_ds.sel(experiment=experiment)
        charts.lines_grid(experiment_zonal_ds[zonal_columns].to_dataarray('column'), 'lat', 'model', 'column',
                          axes=axes, color=FORCING_COLORS[experiment], linewidth=1.6)
        charts.significant_dots_grid(experiment_zonal_ds['mean_change'], experiment_zonal_ds['mean_pvalue'] < ALPHA,
                                     'lat', 'model', None, axes=axes[:, 0], color=FORCING_COLORS[experiment])
        charts.significant_dots_grid(experiment_zonal_ds['width_change'], experiment_zonal_ds['width_pvalue'] < ALPHA,
                                     'lat', 'model', None, axes=axes[:, 3], color=FORCING_COLORS[experiment])
    if ice_edges_da is not None:
        for experiment in ['hist-nat', *FORCED_EXPERIMENTS]:
            zp.ice_edge_lines_grid(ice_edges_da.sel(model=zonal_models, experiment=experiment, season=season).expand_dims(
                column=zonal_columns), 'model', 'column', axes=axes, color=FORCING_COLORS[experiment])

    for ax, column in zip(axes[0], zonal_columns):
        ax.set_title(f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}', loc='left')
    for ax, model in zip(axes[:, 0], zonal_models):
        ax.set_ylabel(f'{model}\n({VARIABLE.units})')
    for ax in axes[-1]:
        ax.set_xlabel('Latitude')
    axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
    axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
    extremes_ylim = (min(ax.get_ylim()[0] for ax in axes[0, 1:]), max(ax.get_ylim()[1] for ax in axes[0, 1:]))
    for ax in axes[0, 1:]:
        ax.set_ylim(extremes_ylim)
    fig.legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS) + zonal_legend_handles,
               loc='outside lower center',
               ncols=len(FORCED_EXPERIMENTS) + len(zonal_legend_handles), frameon=False)
    fig.suptitle(f'Zonal-mean change, final {WINDOW} years against hist-nat, every model: {season}', x=0.01, ha='left')

### 7.4 Every model, by forcing

The same the other way round: one row per experiment and the models as lines, to see how far the models agree on each forcing; one figure per season.

In [ ]:
#(t): The same the other way round, one figure per season: experiments down, one line per model (its shade of the
#(t): experiment's colour, its line style), with each model's ice edge in the experiment's final years
for season in SEASONS:
    season_zonal_ds = zonal_ds.sel(season=season, experiment=FORCED_EXPERIMENTS)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(zonal_columns),
                                                      n_legends=1,
                                                      panel_height=1.95), layout='constrained')
    axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(zonal_columns)).subplots(sharex=True, sharey='col',
                                                                                  squeeze=False)
    for ax in axes.ravel():
        ax.axhline(0, color='0.64', linewidth=0.8, zorder=1)
    for i, experiment in enumerate(FORCED_EXPERIMENTS):
        model_styles = charts.model_lines(experiment, zonal_models)
        for model in zonal_models:
            model_zonal_ds = season_zonal_ds.sel(experiment=experiment, model=model)
            model_color = model_styles[model]['color']
            charts.lines_grid(model_zonal_ds[zonal_columns].to_dataarray('column'), 'lat', None, 'column',
                              axes=axes[i], **model_styles[model])
            charts.plot_significant_dots(model_zonal_ds['mean_change'], model_zonal_ds['mean_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 0], color=model_color)
            charts.plot_significant_dots(model_zonal_ds['width_change'], model_zonal_ds['width_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 3], color=model_color)
            if ice_edges_da is not None:
                zp.ice_edge_lines_grid(ice_edges_da.sel(model=model, experiment=experiment, season=season).expand_dims(
                    column=zonal_columns), None, 'column', axes=axes[i], color=model_color)

    for ax, column in zip(axes[0], zonal_columns):
        ax.set_title(f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}', loc='left')
    for ax, experiment in zip(axes[:, 0], FORCED_EXPERIMENTS):
        ax.set_ylabel(f'{experiment}\n({VARIABLE.units})')
    for ax in axes[-1]:
        ax.set_xlabel('Latitude')
    axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
    axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
    extremes_ylim = (min(ax.get_ylim()[0] for ax in axes[0, 1:]), max(ax.get_ylim()[1] for ax in axes[0, 1:]))
    for ax in axes[0, 1:]:
        ax.set_ylim(extremes_ylim)
    #(c): No hist-nat band in this layout: the rows are experiments. One legend for every row: model k is the k-th
    #(c): shade of each row's colour, shown in historical's greys
    legend_styles = charts.model_lines('historical', zonal_models)
    fig.legend(handles=[Line2D([], [], **style) for style in legend_styles.values()]
               + [zonal_legend_handles[0], zonal_legend_handles[2]], loc='outside lower center',
               ncols=len(zonal_models) + 2, frameon=False)
    fig.suptitle(f'Zonal-mean change by forcing, every model as a line: {season}', x=0.01, ha='left')

### 7.5 Where the seasons vary most

hist-nat's own width $Q_{95} - Q_{05}$ (final 21 years), zonally averaged, with each model's hist-nat ice edge. The paper finds the year-to-year variability of winter temperature largest just poleward of the ice edge, where a season can be over ice or open water; a model whose peak sits elsewhere, or whose edge is far from the others', will put its changes in variability at other latitudes too.

In [ ]:
#(t): hist-nat's zonal-mean width Q95 − Q05 by latitude, one line per model (its shade of hist-nat's green, its line
#(t): style), one panel per season, with each model's hist-nat ice edge (dashed)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=1), layout='constrained')
axes = fig.add_gridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
hist_nat_styles = charts.model_lines('hist-nat', zonal_models)
for model in zonal_models:
    charts.lines_grid(zonal_ds['hist_nat_width'].sel(model=model, season=SEASONS), 'lat', None, 'season', axes=axes,
                      **hist_nat_styles[model])
    if ice_edges_da is not None:
        zp.ice_edge_lines_grid(ice_edges_da.sel(model=model, experiment='hist-nat', season=SEASONS), None, 'season',
                               axes=axes, color=hist_nat_styles[model]['color'])

for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
    ax.set_xlabel('Latitude')
axes[0, 0].set_ylabel(f'hist-nat Q95 − Q05 ({VARIABLE.units})')
axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
axes[0, 0].set_ylim(0, None)
fig.legend(handles=[Line2D([], [], **style) for style in hist_nat_styles.values()]
           + [zonal_legend_handles[2]],
           loc='outside lower center', ncols=len(zonal_models) + 1, frameon=False)
fig.suptitle('Year-to-year variability in hist-nat: zonal-mean width, every season', x=0.01, ha='left');

### 7.6 Sea ice itself

Each experiment's extent through time (ensemble mean and the 5–95% of members), and the equivalent latitude of each model's edge in the final years.

In [ ]:
#(t): Each experiment's extent: the ensemble mean and the 5-95% of members
ice_extent_summary_ds = sea_ice.extent_summary(ice_extents_tree)
ice_extent_summary_ds

In [ ]:
#(t): Each experiment's extent through time (million km²): its ensemble mean (line) and its members' 5-95% (band),
#(t): seasons down and models across; the y axis shared along each season's row
extent_experiments = ['hist-nat'] + FORCED_EXPERIMENTS
extent_models = list(ice_extent_summary_ds['model'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(SEASONS), n_cols=len(extent_models), n_legends=1),
                 layout='constrained')
axes = fig.add_gridspec(len(SEASONS), len(extent_models)).subplots(sharex=True, sharey='row', squeeze=False)

for experiment in extent_experiments:
    experiment_extent_ds = ice_extent_summary_ds.sel(experiment=experiment, season=SEASONS) / 1e6
    charts.bands_grid(experiment_extent_ds['low'], experiment_extent_ds['high'], 'year', 'season', 'model', axes=axes,
                      color=FORCING_COLORS[experiment], alpha=0.15)
    charts.lines_grid(experiment_extent_ds['mean'], 'year', 'season', 'model', axes=axes,
                      color=FORCING_COLORS[experiment], linewidth=1.4)

for ax, model in zip(axes[0], extent_models):
    ax.set_title(model, loc='left')
for ax, season in zip(axes[:, 0], SEASONS):
    ax.set_ylabel(f'{season}\nExtent (million km²)')
fig.legend(handles=charts.forcing_legend_handles(extent_experiments)
           + [Patch(color='0.6', alpha=0.4, label='5–95% of members')],
           loc='outside lower center', ncols=len(extent_experiments) + 1, frameon=False)
fig.suptitle('Antarctic sea-ice extent (cells with at least 15% ice), every model, experiment and season', x=0.01,
             ha='left');

In [ ]:
#(t): The equivalent latitude of each model's ice edge in the final years
ice_edges_da.to_dataframe().unstack('season')['edge_latitude'].round(1)

### 7.7 Change in sea ice

Each experiment's ensemble-mean concentration over the final `WINDOW` years minus hist-nat's (notebook calc_04, section 7), with hist-nat's edge (dashed) and the experiment's (solid). Blue: more ice; red: less. The same four figures as the other sections, but with no test: what goes in, one model, every model, and the multi-model median.

In [ ]:
#(t): EXAMPLE_MODEL's concentration over the final WINDOW years, every experiment
model_season_ice_da = model_ice_da.sel(season=SEASONS)
model_season_ice_da

In [ ]:
#(t): What goes in: hist-nat's and EXPERIMENT's concentration, each with its edge (one colour bar), then the change
#(t): with both edges
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
#(c): The concentration's colour bar under rows 1 and 2, the change's under row 3
concentration_cax = antarctica.colorbar_axis_below(axes[:2])
change_cax = antarctica.colorbar_axis_below(axes[2])

#(c): Rows 1 and 2: hist-nat's and EXPERIMENT's concentration, each with its own edge
antarctica.heatmaps_of_antarctica(model_season_ice_da.sel(experiment=['hist-nat', EXPERIMENT]), 'experiment', 'season',
                                  axes=axes[:2], cax=concentration_cax, **concentration_scale.colours())
antarctica.ice_edges(model_season_ice_da.sel(experiment='hist-nat'), None, 'season', axes=axes[0], linestyle='dashed')
antarctica.ice_edges(model_season_ice_da.sel(experiment=EXPERIMENT), None, 'season', axes=axes[1])
#(c): Row 3: the change, with both edges
antarctica.heatmaps_of_antarctica(ice_change_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS), None,
                                  'season',
                                  axes=axes[2], cax=change_cax, **concentration_change_scale.colours())
antarctica.ice_edges(model_season_ice_da.sel(experiment='hist-nat'), None, 'season', axes=axes[2], linestyle='dashed')
antarctica.ice_edges(model_season_ice_da.sel(experiment=EXPERIMENT), None, 'season', axes=axes[2])

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [f'hist-nat\nfinal {WINDOW} years', f'{EXPERIMENT}\nfinal {WINDOW} years',
                             f'Change\n{EXPERIMENT} − hist-nat'])
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'What goes into the change in sea ice: {EXAMPLE_MODEL}, {EXPERIMENT}');

In [ ]:
#(t): One model: EXAMPLE_MODEL's change in every experiment (rows), with hist-nat's edge (dashed) and each experiment's
#(t): (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(ice_change_da.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                    season=SEASONS), 'experiment',
                                  'season', axes=axes, cax=cax, **concentration_change_scale.colours())
antarctica.ice_edges(model_season_ice_da.sel(experiment='hist-nat',
                                             drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                     'experiment', 'season', axes=axes, linestyle='dashed')
antarctica.ice_edges(model_season_ice_da.sel(experiment=FORCED_EXPERIMENTS), 'experiment', 'season', axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: change in sea-ice concentration, final {WINDOW} years against hist-nat');

In [ ]:
#(t): Every model (rows), one figure per experiment, with hist-nat's edge (dashed) and the experiment's (solid)
models = ice_change_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(ice_change_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                      axes=axes, cax=cax, **concentration_change_scale.colours())
    antarctica.ice_edges(ice_climatology_da.sel(experiment='hist-nat', season=SEASONS), 'model', 'season', axes=axes,
                         linestyle='dashed')
    antarctica.ice_edges(ice_climatology_da.sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'Change in sea-ice concentration, final {WINDOW} years against hist-nat: {experiment}')

In [ ]:
#(t): The multi-model median of the change
median_ice_change_da = ice_change_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS).median('model')
median_ice_change_da

In [ ]:
#(t): The multi-model median, every experiment (rows), with the edges of the models' median concentration
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(median_ice_change_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **concentration_change_scale.colours())
antarctica.ice_edges(median_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                       drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                     'experiment', 'season', axes=axes, linestyle='dashed')
antarctica.ice_edges(median_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                     axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Change in sea-ice concentration, final {WINDOW} years against hist-nat, multi-model median');

### 7.8 Year-to-year correlation

In hist-nat, the correlation over every member and year between a cell's seasonal concentration and its seasonal temperature (notebook calc_04, section 7); ERA5's, from its detrended seasons over 1979–2013, in the last row. Strongly negative where a season with more ice is a colder season: that is where the ice sets the year-to-year variability of the temperature. calc_04 works it out from the full data only.

In [ ]:
#(t): The correlations, every model and ERA5
coupling_rows_da = xr.open_zarr(ICE_RESULTS_DIR / 'ice_tas_correlation.zarr')['correlation'].load()

In [ ]:
#(t): ERA5's concentration over 1979-2013 (%), for its edge
era5_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'era5_climatology.zarr')['siconc'].load()

In [ ]:
#(t): Each row's own edge: hist-nat's final years for the models, ERA5's 1979-2013, in the rows' order
coupling_ice_da = xr.concat([ice_climatology_da.sel(experiment='hist-nat', drop=True),
                             era5_climatology_da.expand_dims(model=['ERA5'])], dim='model')
coupling_ice_da = coupling_ice_da.sel(model=coupling_rows_da['model'])
coupling_ice_da

In [ ]:
#(t): Every model and ERA5 (rows) and season (columns): the correlation, with each row's own edge
n_rows = coupling_rows_da.sizes['model']
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=n_rows, n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=n_rows, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(coupling_rows_da.sel(season=SEASONS), 'model', 'season', axes=axes, cax=cax,
                                  **correlation_scale.colours())
antarctica.ice_edges(coupling_ice_da.sel(season=SEASONS), 'model', 'season', axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, coupling_rows_da['model'].values)
fig.suptitle('Year to year, how the sea ice and the temperature move together: hist-nat, and ERA5');